# PPO on Colab GPU — single machine, GPU rollout (no VM)

Run **everything on the Colab GPU**, and **vectorize the rollout**: instead of B=1 (one game at a
time, which leaves the GPU idle), `rollout_worker --num-envs 32` drives **32 games in lockstep** and
does **one `(B,…)` GPU forward per step** — the policy-server pattern. orbit_wars is simultaneous-move,
so all envs step together cleanly. **32 episodes/iter**, **mixed 2P/4P**, fresh seeds each iteration.

Loop: `rollout_worker --device cuda --num-envs 32` (writes shards to GCS) → `learner_step` (updates,
pushes `policy_v{K+1}`) → repeat. The env (CPU game engine) becomes the bottleneck before the GPU
does — that's the point: the GPU goes from 32 sequential forwards/step to 1.

Run §0 → §6 top to bottom.

## 0. Config

In [ ]:
import time
PROJECT = 'analog-receiver-489214-e9'   # gcloud needs a project; Colab auth sets none
BUCKET  = 'gs://orbit-wars-shipping'
RUN_ID  = 'ppo_gpu_' + time.strftime('%Y%m%d-%H%M%S')
PREFIX  = f'{BUCKET}/ppo/{RUN_ID}'

HISTORY_WINDOW = 10            # T=10 rollout window (GPU makes this affordable)
ITERS          = 5
EPISODES       = 32            # self-play games per iter
NUM_ENVS       = 32            # BATCHED: run this many games in lockstep -> ONE (B,...) GPU
                               # forward per step (vectorized). =EPISODES runs all in one batch.
NUM_PLAYERS    = 'mix'         # 'mix' = alternate 2P/4P (distinct seeds); or '2' / '4'
SEED_BASE      = 100000        # iter K uses seeds [SEED_BASE + K*EPISODES ..]  -> fresh games each iter
DEVICE         = 'cuda'        # rollout + learner both on the GPU
# Rollout shard tensor caps. NOTE: episodes_to_ppo stacks ALL steps of the iter
# before minibatching, so 32 eps x T=10 is memory-heavy on the GPU. If learner_step
# OOMs: drop EPISODES to ~16, or MAX_FLEETS to 256/512 (actual fleets << 1024, so
# this is safe), or run the learner on CPU RAM (LEARNER_DEVICE='cpu', slower).
MAX_PLANETS=64; MAX_FLEETS=1024; LEARNER_DEVICE=DEVICE

# frozen base (actor); critic = glob value head trained in PPO (actor has no consolidator)
BASELINE_RUN = 'L3L4_T10_film3_head3_d256_b256_20ep_lr0.0001_20260529-084115'
ENTITY_CKPT  = f'{BUCKET}/entity/runs/{BASELINE_RUN}/entity_encoder_best.pt'
PAIR_CACHE_PREFIX = ''         # BC off (no T=10 pair cache); else a matching prefix

# PPO hyperparams
MINIBATCH=256; EPOCHS=3; CLIP=0.10; TARGET_KL=0.01; LR_HEADS=1e-4
VALUE_COEF=0.5; ENT_COEF=0.01; SIGMA=0.35; BC_COEF=0.05
VM_ID='colab'                  # shard subdir name (this machine)
print('RUN_ID', RUN_ID, '| T', HISTORY_WINDOW, '| iters', ITERS, '| episodes', EPISODES,
      '| players', NUM_PLAYERS, '| device', DEVICE)

## 1. Authenticate + set project

In [ ]:
from google.colab import auth
auth.authenticate_user()
import subprocess
subprocess.run(['gcloud','config','set','project',PROJECT], check=True)   # REQUIRED for any gcloud
def gcs(*a): return subprocess.run(['gcloud','storage',*a], check=True, capture_output=True, text=True)
def gcs_exists(url):
    try: gcs('objects','describe',url,'--format=value(size)'); return True
    except subprocess.CalledProcessError: return False
print('project set:', PROJECT)

## 2. Stage code + weights + actor ckpt

In [ ]:
import os, shutil, sys
subprocess.run([sys.executable,'-m','pip','install','-q','kaggle_environments','psutil'], check=True)
from pathlib import Path
WORK = Path('/content/orbit-wars'); WORK.mkdir(parents=True, exist_ok=True); os.chdir(WORK)
for rel in ('agents','scripts','ckpts'): shutil.rmtree(WORK/rel, ignore_errors=True)
# entity/ has the FLAT weights.tgz (planet/fleet/comet) + the ppo-package code.tgz
gcs('cp', f'{BUCKET}/entity/code.tgz', '.');    subprocess.run(['tar','xzf','code.tgz'], check=True)
gcs('cp', f'{BUCKET}/entity/weights.tgz', '.'); subprocess.run(['tar','xzf','weights.tgz'], check=True)
PLANET_DIR=WORK/'ckpts/planet'; FLEET_DIR=WORK/'ckpts/fleet'; COMET_DIR=WORK/'ckpts/comet'; ENT_DIR=WORK/'ckpts/entity'
for d in (PLANET_DIR,FLEET_DIR,COMET_DIR,ENT_DIR): d.mkdir(parents=True, exist_ok=True)
for src,dst in (('planet_encoder_best.pt',PLANET_DIR),('fleet_encoder_best.pt',FLEET_DIR),('comet_past_best.pt',COMET_DIR)):
    if (WORK/src).exists(): shutil.copy(WORK/src, dst/src)
ENTITY_LOCAL=str(ENT_DIR/'entity_encoder_best.pt'); gcs('cp', ENTITY_CKPT, ENTITY_LOCAL)
for m in [m for m in sys.modules if m.startswith('agents')]: del sys.modules[m]
import agents; from agents.transformer_v2.ppo import shards
import torch; print('agents at', agents.__file__, '| cuda:', torch.cuda.is_available())

## 3. Build + push `policy_v0` (full ckpt + head-delta)

In [ ]:
import torch, json
from agents.transformer_v2.ppo.smoke import load_supervised
from agents.transformer_v2.ppo.actor_critic import PPOActorCritic
entity_model, fleet_enc, planet_enc, comet_enc, cfg = load_supervised(
    Path(ENTITY_LOCAL), DEVICE, planet_run_dir=PLANET_DIR, fleet_run_dir=FLEET_DIR, comet_run_dir=COMET_DIR)
print('actor PlayerConsolidator:', getattr(entity_model,'consolidator',None) is not None,
      '-> critic = glob value_head (trained in PPO)')
policy = PPOActorCritic(entity_model, sigma=SIGMA, allow_debug_glob_critic=True).to(DEVICE)
print('freeze groups:', policy.freeze_for_phase(0))
BASE_VERSION = shards.sha256_file(ENTITY_LOCAL)[:12]
torch.save({'policy':policy.state_dict(),'value_hidden':policy.value_head[0].out_features,
            'sigma':SIGMA,'iter':0}, 'policy_v0.pt')
dinfo = shards.save_trainable_delta(policy,'policy_v0.heads.pt', base_version=BASE_VERSION)
gcs('cp','policy_v0.pt',f'{PREFIX}/checkpoints/policy_v0.pt')
gcs('cp','policy_v0.heads.pt',f'{PREFIX}/heads/policy_v0.heads.pt')
Path('config.json').write_text(json.dumps({'run_id':RUN_ID,'history_window':HISTORY_WINDOW,
    'base_version':BASE_VERSION,'episodes':EPISODES,'num_players':NUM_PLAYERS,'device':DEVICE}, indent=2))
gcs('cp','config.json',f'{PREFIX}/config.json')
del policy, entity_model; torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('pushed policy_v0', dinfo, 'base', BASE_VERSION)

## 4. The loop — GPU rollout (32 eps, mixed 2P/4P) → learner_step

Each iteration: `rollout_worker --device cuda` rolls 32 self-play games (alternating 2P/4P, fresh
seeds), uploads shards to `rollouts/vK/colab/`; then `learner_step --device cuda` consumes them and
pushes `policy_v{K+1}`. Single process, single worker — no VM, no daemon.

In [ ]:
import re

def run_streaming(cmd):
    # subprocess.run's inherited stdout does NOT stream into a Colab cell -- it
    # surfaces only when the child exits, so a long rollout looks "stuck". Read
    # the child's stdout via a pipe and re-print each line at the Python level so
    # the rollout heartbeats + learner signals show LIVE.
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)

def latest_policy_version():
    out = subprocess.run(['gcloud','storage','ls',f'{PREFIX}/checkpoints/'],capture_output=True,text=True).stdout
    vs=[int(m.group(1)) for m in re.finditer(r'policy_v(\d+)\.pt', out)]; return max(vs) if vs else 0

HP = ['--minibatch-size',str(MINIBATCH),'--epochs',str(EPOCHS),'--clip',str(CLIP),
      '--target-kl',str(TARGET_KL),'--lr-heads',str(LR_HEADS),'--value-coef',str(VALUE_COEF),
      '--ent-coef',str(ENT_COEF),'--sigma',str(SIGMA)]
ENC = ['--fleet-run-dir',str(FLEET_DIR),'--planet-run-dir',str(PLANET_DIR),'--comet-run-dir',str(COMET_DIR)]

for K in range(latest_policy_version(), ITERS):
    print(f'\n========== iter {K} :: GPU rollout ({EPISODES} eps, {NUM_PLAYERS}) ==========', flush=True)
    run_streaming(['python','-u','-m','agents.transformer_v2.ppo.rollout_worker',
        '--ckpt',ENTITY_LOCAL, *ENC,
        '--policy-heads',f'{PREFIX}/heads/policy_v{K}.heads.pt','--base-version',BASE_VERSION,
        '--policy-version',str(K),'--history-window',str(HISTORY_WINDOW),
        '--episodes',str(EPISODES),'--num-players',NUM_PLAYERS,'--num-envs',str(NUM_ENVS),'--rollout-workers','1',
        '--max-planets',str(MAX_PLANETS),'--max-fleets',str(MAX_FLEETS),
        '--device',DEVICE,'--out',f'{PREFIX}/rollouts/v{K}/{VM_ID}/','--vm-id',VM_ID,
        '--seed-base',str(SEED_BASE + K*EPISODES),'--progress-every','10','--verbose'])
    print(f'---------- iter {K} :: learner_step -> v{K+1} ----------', flush=True)
    cmd=['python','-u','-m','agents.transformer_v2.ppo.learner_step',
        '--policy-ckpt',f'{PREFIX}/checkpoints/policy_v{K}.pt','--shards',f'{PREFIX}/rollouts/v{K}/',
        '--out-ckpt',f'{PREFIX}/checkpoints/policy_v{K+1}.pt','--out-heads',f'{PREFIX}/heads/policy_v{K+1}.heads.pt',
        '--train-log',f'{PREFIX}/train_log.jsonl','--policy-version',str(K),'--base-version',BASE_VERSION,
        '--device',LEARNER_DEVICE,'--ckpt',ENTITY_LOCAL, *ENC, *HP]
    if PAIR_CACHE_PREFIX: cmd += ['--pair-cache-path',str(WORK/'pair_cache.pt'),'--bc-coef',str(BC_COEF)]
    run_streaming(cmd)
print('\nloop complete')

## 5. Inspect — PPO training signals + game status (per iter)

In [ ]:
import json
print('=== PPO TRAINING SIGNALS (per iter, from train_log.jsonl) ===')
raw = subprocess.run(['gcloud','storage','cat',f'{PREFIX}/train_log.jsonl'],capture_output=True,text=True).stdout
for ln in raw.splitlines():
    r=json.loads(ln)
    print(f"iter {r['iter']:>2}: winrate={r.get('winrate')}  kl={r.get('avg_kl')}  "
          f"policy_loss={r.get('policy_loss')}  value_loss={r.get('value_loss')}  "
          f"entropy={r.get('entropy')}  clip_frac={r.get('clip_frac')}  "
          f"(eps={r.get('n_episodes')} steps={r.get('n_steps')} wall={r.get('wall_s')}s)")

print('\n=== GAME STATUS (per iter: win rate, 2P/4P split, game length, RAM/timing) ===')
for K in range(ITERS):
    m = shards.read_progress(f'{PREFIX}/rollouts/v{K}/{VM_ID}/metrics.json')
    if not m: continue
    pe = m.get('per_episode',[])
    g2=[e for e in pe if e['num_players']==2]; g4=[e for e in pe if e['num_players']==4]
    w2=sum(e['won'] for e in g2); w4=sum(e['won'] for e in g4)
    steps=[e['steps'] for e in pe] or [0]
    split=[]
    if g2: split.append(f"2P {w2}/{len(g2)}={w2/len(g2)*100:.0f}%")
    if g4: split.append(f"4P {w4}/{len(g4)}={w4/len(g4)*100:.0f}%")
    print(f"v{K}: {len(pe)} games, {m.get('n_wins')}/{len(pe)} wins  [{', '.join(split)}]  "
          f"game-len mean {sum(steps)/len(steps):.0f} (min {min(steps)} max {max(steps)})  | "
          f"peak_rss {m.get('peak_rss_mb')}MB  mean_step {m.get('mean_step_s')}s  wall {m.get('total_wall_s')}s")